# Module 1: RNA-seq Preprocessing and Clustering

**Scripts covered:** `1_rna_analysis/preprocessing_zhao.py`, `1_rna_analysis/rnaseq_clustering.py`

## Purpose

Before building the VCell reaction-diffusion model, we need to set biologically realistic initial conditions for the protein concentrations. This module processes multi-source RNA-seq data from breast cancer cell lines to:

1. Normalize raw read counts to **TPM** (Transcripts Per Million)
2. Apply **log2 transformation** and **z-score normalization**
3. Run **PCA** and **hierarchical clustering** to understand expression patterns across datasets
4. Output normalized expression tables for downstream protein abundance inference (Module 2)

### Data sources

| Dataset | Description | Cell type |
|---------|-------------|----------|
| Zhao et al. (TNBC) | Triple-negative breast cancer patient biopsies | Tumor tissue |
| TCGA breast cancer | The Cancer Genome Atlas bulk RNA-seq | Tumor tissue |
| MCF10A (CCLE) | Non-tumorigenic mammary epithelial cell line | Cell line |
| Kang et al. 2013 | GSM1100205/206 breast cancer lines | Cell line |

### Genes of interest

We focus on 15 genes in the CPC regulatory network:

```
AURKB, BIRC5, BUB1, CASC5, CDCA8, GSG2, INCENP, KAT5,
NDC80, PLK1, HASPIN, KNL1, SGO1, SGOL1, TTK
```

These encode the core CPC subunits and their key interactors at the kinetochore/centromere.


## Setup

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Update these paths to match your local setup
repo = "/path/to/VCell_Analysis"
indir = "/path/to/Box/CPC_Model_Project/VCell_RNAseq/Zhao_dataset/"

## Part 1: Loading the Zhao dataset

The Zhao dataset (`RQ021672-Zhao_counts.csv`) contains **raw read counts** from TNBC patient tumor biopsies. The rows are Ensembl gene IDs (e.g., `ENSG00000...`) and columns are sample identifiers.

We also load a metadata file that maps sample IDs to clinical annotations (race, tumor subtype, etc.).

In [ ]:
# Load raw read counts — rows are gene IDs, columns are samples
zhao_data = pd.read_csv(f"{indir}RQ021672-Zhao_counts.csv", index_col=0, header=0)
print(f"Shape: {zhao_data.shape}  (genes x samples)")
zhao_data.head()

In [ ]:
# Load metadata for sample annotations
zhao_metadata = pd.read_excel(
    f"{indir}/Copy of DEID_TNBC Shavings Annotation.xlsx",
    sheet_name='Sheet1', index_col=0
)
zhao_metadata.head()

## Part 2: TPM Normalization

Raw read counts are not directly comparable across samples because they depend on sequencing depth and gene length. **TPM (Transcripts Per Million)** corrects for both.

### Formula

1. **Reads Per Kilobase (RPK)**: divide each gene's count by its length in kb
   $$\text{RPK}_g = \frac{\text{count}_g}{\text{length}_g \text{ (kb)}}$$

2. **TPM**: normalize RPK so each sample sums to 1,000,000
   $$\text{TPM}_g = \frac{\text{RPK}_g}{\sum_i \text{RPK}_i} \times 10^6$$

Gene lengths come from `ncbi_ensembl_coding_mergedgenelength.csv`, which was built from NCBI/Ensembl annotations.

In [ ]:
def tpm_normalization(df, genelengths):
    """
    Converts raw read counts to TPM.
    
    Parameters
    ----------
    df : pd.DataFrame
        Genes x samples matrix of raw counts. Index should be Ensembl gene IDs.
    genelengths : pd.DataFrame or pd.Series
        Gene lengths in base pairs, indexed by Ensembl ID.
    
    Returns
    -------
    pd.DataFrame
        TPM-normalized expression matrix, same shape as input.
    """
    df = df.copy()
    # Strip version numbers from Ensembl IDs (e.g., ENSG00000001234.5 -> ENSG00000001234)
    df['gene_id'] = df.index.str.split('.').str[0]

    # Get a length Series from whatever format was passed
    if isinstance(genelengths, pd.DataFrame):
        length_series = genelengths['length'] if 'length' in genelengths.columns else genelengths.iloc[:, 0]
    else:
        length_series = genelengths

    # Handle duplicate gene IDs by averaging their lengths
    if length_series.index.duplicated().any():
        length_series = length_series.groupby(level=0).mean()

    # Map lengths (convert bp -> kb)
    df['length_kb'] = df['gene_id'].map(length_series) / 1000.0
    df = df.dropna(subset=['length_kb'])  # drop genes with no length info

    counts = df.drop(columns=['gene_id', 'length_kb'])

    # Step 1: RPK
    rpk = counts.div(df['length_kb'], axis=0)

    # Step 2: TPM
    per_sample_sum = rpk.sum(axis=0)
    per_sample_sum[per_sample_sum == 0] = np.nan  # avoid division by zero
    tpm = rpk.div(per_sample_sum, axis=1) * 1e6

    return tpm

In [ ]:
# Load gene lengths
genelengths = pd.read_csv(
    f"{repo}/1_rna_analysis/data/ncbi_ensembl_coding_mergedgenelength.csv",
    index_col=0, header=0, sep='\t'
)

# Run TPM normalization
# Note: zhao_data.iloc[:, 1:] skips the first column, which contains gene names (not counts)
zhao_tpm = tpm_normalization(zhao_data.iloc[:, 1:], genelengths=genelengths)
print(f"TPM matrix shape: {zhao_tpm.shape}")

# Verify: each sample should sum to ~1,000,000
print(f"\nSample sum range: [{zhao_tpm.sum().min():.0f}, {zhao_tpm.sum().max():.0f}]")

## Part 3: Log2 Transformation and Gene Name Mapping

TPM values span several orders of magnitude, so we apply a **log2(TPM + 1)** transformation (the `+1` is a pseudocount to handle zero-expressed genes without taking log of zero). We then replace the Ensembl index with human-readable gene names from the `genename` column in the original data.

In [ ]:
zhao_logtpm = np.log2(zhao_tpm + 1)

# Map back to gene names (stored in the first column of zhao_data)
zhao_logtpm['genename'] = zhao_data['genename']
zhao_logtpm = zhao_logtpm.set_index('genename')

print(f"Log2 TPM matrix shape: {zhao_logtpm.shape}")
zhao_logtpm.head()

In [ ]:
# Save full log2 TPM table for reference
zhao_logtpm.to_csv(f"{repo}/1_rna_analysis/data/zhao_logTPM_allgenes.csv")

## Part 4: Subsetting to Network Genes

We extract only the 15 CPC network genes. The resulting matrix is transposed so that rows are samples and columns are genes — the format expected by downstream clustering and PCA.

In [ ]:
genes_of_interest = [
    'AURKB', 'BIRC5', 'BUB1', 'CASC5', 'CDCA8', 'GSG2', 'INCENP',
    'KAT5', 'NDC80', 'PLK1', 'HASPIN', 'KNL1', 'SGO1', 'SGOL1', 'TTK'
]

zhao_subset = zhao_logtpm[zhao_logtpm.index.isin(genes_of_interest)]
zhao_subset = zhao_subset.transpose()  # samples x genes

# Add race metadata for coloring plots later
races = []
for col in zhao_subset.index:
    match = False
    for casenum in zhao_metadata.index:
        if casenum in col:
            races.append(zhao_metadata.loc[casenum]['race'])
            match = True
            break
    if not match:
        races.append('Unknown')

zhao_subset['race'] = races
zhao_subset.to_csv(f"{repo}/1_rna_analysis/data/zhao_logTPM_network_genes.csv")
print(f"Subset shape: {zhao_subset.shape}  (samples x genes + metadata)")

## Part 5: Z-score Normalization

For PCA and clustering, we z-score normalize across samples for each gene — this removes differences in absolute expression level between genes and focuses the analysis on **relative patterns** across samples.

$$z_g = \frac{x_g - \mu_g}{\sigma_g}$$

`StandardScaler` from scikit-learn does this per column (i.e., per gene).

In [ ]:
data = zhao_subset.drop(columns=['race'])
scaler = StandardScaler()

# fillna(0) handles any genes with missing expression in some samples
normalized_data = pd.DataFrame(
    scaler.fit_transform(data.fillna(0)),
    index=data.index,
    columns=data.columns
)
normalized_data['race'] = races
normalized_data.to_csv(f"{repo}/1_rna_analysis/data/zhao_logTPM_network_genes_zscore.csv")
normalized_data.head()

## Part 6: PCA

PCA reduces the 15-gene expression space to two principal components that capture the most variance across samples. This lets us visualize whether samples cluster by tumor subtype, treatment, race, or other factors.

In [ ]:
pca = PCA(n_components=2)
pca_result = pca.fit_transform(normalized_data.drop(columns=['race']))

print(f"PC1 explains {pca.explained_variance_ratio_[0]*100:.1f}% of variance")
print(f"PC2 explains {pca.explained_variance_ratio_[1]*100:.1f}% of variance")

In [ ]:
pca_df = pd.DataFrame(pca_result, columns=['PC1', 'PC2'], index=normalized_data.index)
pca_df['race'] = zhao_subset['race']

fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(x='PC1', y='PC2', hue='race', data=pca_df, s=100, edgecolor='k', ax=ax)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
ax.set_title("PCA of Zhao TNBC Data — CPC network genes")
plt.tight_layout()
plt.show()

## Part 7: Hierarchical Clustering

`1_rna_analysis/rnaseq_clustering.py` extends this analysis to **multiple datasets** (TCGA, CCLE, MCF10A, Kang, Zhao) combined into a single matrix. Hierarchical clustering is used to:

- Identify which cell lines/tumor samples share similar CPC expression profiles
- Determine which tissue types cluster together
- Assign samples to clusters for downstream analysis

The key idea: samples with similar CPC network expression patterns likely share similar centromere biology, which informs which protein abundance ratios to use as VCell initial conditions.

### Running the clustering script

```bash
cd 1_rna_analysis/
python rnaseq_clustering.py
```

This produces:
- `figures/rnaseq_hierarchicalclustering_plot.pdf` — dendrogram with cluster assignments
- `figures/rnaseq_pca_plot.png` — PCA with all datasets overlaid
- `figures/rnaseq_pca_biplot.png` — PCA biplot showing gene loadings
- `data/rnaseq_network_genes_TCGA_MCF10A_CCLE_ZHAO_for_pinferna.csv` — combined table for network inference


## Key outputs and what they feed into

| File | Used by |
|------|---------|
| `zhao_logTPM_allgenes.csv` | Reference; not used downstream directly |
| `zhao_logTPM_network_genes.csv` | Clustering in `rnaseq_clustering.py` |
| `zhao_logTPM_network_genes_zscore.csv` | PCA and clustering |
| `rnaseq_network_genes_TCGA_MCF10A_CCLE_ZHAO_for_pinferna.csv` | Module 2 — protein inference |

The normalized expression values for MCF10A specifically are used to scale the initial protein concentrations in the VCell models, since MCF10A is the primary cell line being modeled.